# Data Preprocessing for Dashboard

In [2]:
# import libraries
import pandas as pd
import numpy as np
import geopandas as gpd

pd.set_option('display.max_columns', None)

In [3]:
# import data
vlgas_df = pd.read_csv('Dataset/Cleaned/cleaned_vlgas.csv')
vwp_df = pd.read_csv('Dataset/Cleaned/cleaned_vwp.csv')

### Fact Table - LGA Waste
1. Extract only the tonnage columns
2. Unpivot wide dataset to long dataset: Categorized columns into channel, category, subcategory and state.

In [4]:
mapping = {
    'garbage_collected_total_tonnes': ('Kerbside', 'Garbage', 'Garbage', 'Collected'),
    'kerbside_recycling_total_collected_tonnes': ('Kerbside', 'Recycling', 'Total', 'Collected'),
    'kerbside_recycling_total_recycled_tonnes': ('Kerbside', 'Recycling', 'Total', 'Recycled'),
    'kerbside_recycling_glass_all_recycled_tonnes': ('Kerbside', 'Recycling', 'Glass', 'Recycled'),
    'kerbside_recycling_metal_recycled_tonnes': ('Kerbside', 'Recycling', 'Metal', 'Recycled'),
    'kerbside_recycling_paper_all_recycled_tonnes': ('Kerbside', 'Recycling', 'Paper', 'Recycled'),
    'kerbside_recycling_plastics_all_recycled_tonnes': ('Kerbside', 'Recycling', 'Plastics', 'Recycled'),
    'kerbside_fogo_collected_tonnes': ('Kerbside', 'FOGO', 'FOGO', 'Collected'),
    'kerbside_go_collected_tonnes':('Kerbside', 'GO', 'GO', 'Collected'),
    'kerbside_fogo_processed_tonnes': ('Kerbside', 'FOGO', 'FOGO', 'Processed'),
    'kerbside_go_processed_tonnes': ('Kerbside', 'GO', 'GO', 'Processed'),
    'dropoff_garbage_collected_tonnes': ('Drop-off', 'Garbage', 'Garbage', 'Collected'),
    'dropoff_recycling_collected_tonnes': ('Drop-off', 'Recycling', 'Total', 'Collected'),
    'kerbside_glass_total_collected_tonnes': ('Kerbside', 'Glass', 'Glass', 'Collected'),
    'hardwaste_collected_tonnes': ('Hardwaste Collection', 'Hard Waste', 'Hard Waste', 'Collected'),
    'hardwaste_disposed_tonnes': ('Hardwaste Collection', 'Hard Waste', 'Hard Waste', 'Disposed')
}

waste_lga = vlgas_df.melt(
    id_vars=['start_year', 'council'],
    value_vars=list(mapping.keys()),
    var_name='waste_type',
    value_name='tonnes'
)

mapping_df = pd.DataFrame.from_dict(
    mapping,
    orient='index',
    columns=['channel', 'category', 'subcategory', 'state']
)

mapping_df.index.name = 'waste_type'

waste_lga = waste_lga.merge(
    mapping_df,
    left_on='waste_type',
    right_index=True,
    how='left'
)

waste_lga = waste_lga.drop(columns=['waste_type'])
waste_lga = waste_lga.dropna(subset=['tonnes'])

In [5]:
# export
waste_lga.to_csv('Dataset/Semantic Model/fact_waste_lga.csv', index=0)

### Dim Table - Council-Year

In [6]:
councilYear_df = vlgas_df[['start_year', 'council', 'population','premises_residential', 'premises_nonresidential', 
                           'premises_with_garbage_service_residential', 'premises_with_garbage_service_nonresidential',
                           'premises_with_kerbside_recycling_service_residential', 'premises_with_kerbside_recycling_service_nonresidential',
                           'premises_with_kerbside_organics_service_residential', 
                           'premises_with_kerbside_glass_service_residential', 'premises_with_kerbside_glass_service_nonresidential',
                           'premises_with_access_to_hardwaste_service_residential',
                           'service_frequency_garbage', 'service_frequency_kerbside_recycling', 
                           'service_frequency_kerbside_organics', 'service_frequency_kerbside_glass', 
                           'has_fogo', 'has_glass_service', 'has_organics_service', 'garbage_is_estimated']]

In [7]:
councilYear_df.to_csv('Dataset/Semantic Model/dim_council_year.csv', index=0)

### Dim Table - Year

In [8]:
year_df = (
    vlgas_df[['financial_year','start_year', 'end_year']]
    .drop_duplicates()
    .sort_values('start_year')
    .reset_index(drop=True)
)

last_year = year_df["start_year"].max()
future_years = list(range(last_year + 1, 2031))
future_df = pd.DataFrame({
    "financial_year": [f"{y}-{y+1}" for y in future_years],
    "start_year": future_years,
    "end_year": [y + 1 for y in future_years]
})

year_df = pd.concat([year_df, future_df], ignore_index=True)
year_df = year_df.sort_values("start_year").reset_index(drop=True)


year_df['decade'] = (year_df['start_year'] // 10) * 10
year_df['fy_short'] = "FY" + year_df['start_year'].astype(str).str[-2:]
year_df['is_forecast'] = np.where(
    year_df['start_year'] > 2023,
    True,
    False
)

year_df

,financial_year,start_year,end_year,decade,fy_short,is_forecast
0,2001-2002,2001,2002,2000,FY01,False
1,2002-2003,2002,2003,2000,FY02,False
2,2003-2004,2003,2004,2000,FY03,False
3,2004-2005,2004,2005,2000,FY04,False
4,2005-2006,2005,2006,2000,FY05,False
5,2006-2007,2006,2007,2000,FY06,False
6,2007-2008,2007,2008,2000,FY07,False
7,2008-2009,2008,2009,2000,FY08,False
8,2009-2010,2009,2010,2000,FY09,False
9,2010-2011,2010,2011,2010,FY10,False


In [9]:
year_df.to_csv('Dataset/Semantic Model/dim_year.csv', index=0)

### Dim Table - Council

In [10]:
council_df = (
    vlgas_df[['council']]
    .drop_duplicates()
    .sort_values('council')
    .reset_index(drop=True)
)

# extract LGA name by removing council type suffixes
council_df['lga'] = council_df['council'].str.replace(
    r'\s*(Rural City Council|Shire Council|City Council|Borough Council)$',
    '',
    regex=True
)

council_type = {
    c: ("Rural City Council" if "Rural City" in c
        else "City Council" if "City" in c
        else "Shire Council" if "Shire" in c
        else "Borough Council")
    for c in council_df['council']
}
council_df['council_type'] = council_df['council'].map(council_type)

council_df['state'] = "VIC"
council_df['country'] = "Australia"

In [11]:
# map LGA to region
region_mapping = {
    'Glenelg': 'Barwon South West',
    'Southern Grampians': 'Barwon South West',
    'Moyne': 'Barwon South West',
    'Corangamite': 'Barwon South West',
    'Colac Otway': 'Barwon South West',
    'Warrnambool': 'Barwon South West',
    'Surf Coast': 'Barwon South West',
    'Greater Geelong': 'Barwon South West',
    'Queenscliffe': 'Barwon South West',
    'Bass Coast': 'Gippsland',
    'Baw Baw': 'Gippsland',
    'East Gippsland': 'Gippsland',
    'South Gippsland': 'Gippsland',
    'Latrobe': 'Gippsland',
    'Wellington': 'Gippsland',
    'Ararat': 'Grampians',
    'Ballarat': 'Grampians',
    'Golden Plains': 'Grampians',
    'Hepburn': 'Grampians',
    'Hindmarsh': 'Grampians',
    'Horsham': 'Grampians',
    'Moorabool': 'Grampians',
    'Northern Grampians': 'Grampians',
    'Pyrenees': 'Grampians',
    'West Wimmera': 'Grampians',
    'Yarriambiack': 'Grampians',
    'Alpine': 'Hume',
    'Benalla': 'Hume',
    'Greater Shepparton': 'Hume',
    'Indigo': 'Hume',
    'Mansfield': 'Hume',
    'Moira': 'Hume',
    'Murrindindi': 'Hume',
    'Mitchell': 'Hume',
    'Strathbogie': 'Hume',
    'Towong': 'Hume',
    'Wangaratta': 'Hume',
    'Wodonga': 'Hume',
    'Buloke': 'Loddon Mallee',
    'Campaspe': 'Loddon Mallee',
    'Central Goldfields': 'Loddon Mallee',
    'Gannawarra': 'Loddon Mallee',
    'Greater Bendigo': 'Loddon Mallee',
    'Loddon': 'Loddon Mallee',
    'Macedon Ranges': 'Loddon Mallee',
    'Mildura': 'Loddon Mallee',
    'Mount Alexander': 'Loddon Mallee',
    'Swan Hill': 'Loddon Mallee',
    'Banyule': 'Port Phillip',
    'Bayside': 'Port Phillip',
    'Boroondara': 'Port Phillip',
    'Brimbank': 'Port Phillip',
    'Cardinia': 'Port Phillip',
    'Casey': 'Port Phillip',
    'Darebin': 'Port Phillip',
    'Frankston': 'Port Phillip',
    'Glen Eira': 'Port Phillip',
    'Greater Dandenong': 'Port Phillip',
    'Hobsons Bay': 'Port Phillip',
    'Hume': 'Port Phillip',
    'Kingston': 'Port Phillip',
    'Knox': 'Port Phillip',
    'Manningham': 'Port Phillip',
    'Maribyrnong': 'Port Phillip',
    'Maroondah': 'Port Phillip',
    'Melbourne': 'Port Phillip',
    'Melton': 'Port Phillip',
    'Merri-Bek': 'Port Phillip',
    'Monash': 'Port Phillip',
    'Moonee Valley': 'Port Phillip',
    'Mornington Peninsula': 'Port Phillip',
    'Nillumbik': 'Port Phillip',
    'Port Phillip': 'Port Phillip',
    'Stonnington': 'Port Phillip',
    'Whitehorse': 'Port Phillip',
    'Whittlesea': 'Port Phillip',
    'Wyndham': 'Port Phillip',
    'Yarra': 'Port Phillip',
    'Yarra Ranges': 'Port Phillip',
}

council_df['region'] = council_df['lga'].map(region_mapping)

In [12]:
council_region_type_mapping = {
    # Metropolitan
    "Banyule City Council":            "Metropolitan",
    "Bayside City Council":            "Metropolitan",
    "Boroondara City Council":         "Metropolitan",
    "Brimbank City Council":           "Metropolitan",
    "Casey City Council":              "Metropolitan",
    "Darebin City Council":            "Metropolitan",
    "Frankston City Council":          "Metropolitan",
    "Glen Eira City Council":          "Metropolitan",
    "Greater Dandenong City Council":  "Metropolitan",
    "Hobsons Bay City Council":        "Metropolitan",
    "Hume City Council":               "Metropolitan",
    "Kingston City Council":           "Metropolitan",
    "Knox City Council":               "Metropolitan",
    "Manningham City Council":         "Metropolitan",
    "Maribyrnong City Council":        "Metropolitan",
    "Maroondah City Council":          "Metropolitan",
    "Melbourne City Council":          "Metropolitan",
    "Melton City Council":             "Metropolitan",
    "Merri-Bek City Council":          "Metropolitan",
    "Monash City Council":             "Metropolitan",
    "Moonee Valley City Council":      "Metropolitan",
    "Mornington Peninsula Shire Council": "Metropolitan",
    "Nillumbik Shire Council":         "Metropolitan",
    "Port Phillip City Council":       "Metropolitan",
    "Stonnington City Council":        "Metropolitan",
    "Whitehorse City Council":         "Metropolitan",
    "Whittlesea City Council":         "Metropolitan",
    "Wyndham City Council":            "Metropolitan",
    "Yarra City Council":              "Metropolitan",
    "Yarra Ranges Shire Council":      "Metropolitan",
    # peri-urban growth corridors — classed as Metropolitan for dashboard purposes
    "Cardinia Shire Council":          "Metropolitan",
    "Mitchell Shire Council":          "Metropolitan",

    # Regional City
    "Ballarat City Council":           "Regional City",
    "Greater Bendigo City Council":    "Regional City",
    "Greater Geelong City Council":    "Regional City",
    "Greater Shepparton City Council": "Regional City",
    "Latrobe City Council":            "Regional City",
    "Mildura Rural City Council":      "Regional City",
    "Warrnambool City Council":        "Regional City",
    "Wangaratta Rural City Council":   "Regional City",
    "Wodonga City Council":            "Regional City",

    # Rural & Regional
    "Alpine Shire Council":            "Rural & Regional",
    "Ararat Rural City Council":       "Rural & Regional",
    "Bass Coast Shire Council":        "Rural & Regional",
    "Baw Baw Shire Council":           "Rural & Regional",
    "Benalla Rural City Council":      "Rural & Regional",
    "Buloke Shire Council":            "Rural & Regional",
    "Campaspe Shire Council":          "Rural & Regional",
    "Central Goldfields Shire Council":"Rural & Regional",
    "Colac Otway Shire Council":       "Rural & Regional",
    "Corangamite Shire Council":       "Rural & Regional",
    "East Gippsland Shire Council":    "Rural & Regional",
    "Gannawarra Shire Council":        "Rural & Regional",
    "Glenelg Shire Council":           "Rural & Regional",
    "Golden Plains Shire Council":     "Rural & Regional",
    "Hepburn Shire Council":           "Rural & Regional",
    "Hindmarsh Shire Council":         "Rural & Regional",
    "Horsham Rural City Council":      "Rural & Regional",
    "Indigo Shire Council":            "Rural & Regional",
    "Loddon Shire Council":            "Rural & Regional",
    "Macedon Ranges Shire Council":    "Rural & Regional",
    "Mansfield Shire Council":         "Rural & Regional",
    "Moira Shire Council":             "Rural & Regional",
    "Moorabool Shire Council":         "Rural & Regional",
    "Mount Alexander Shire Council":   "Rural & Regional",
    "Moyne Shire Council":             "Rural & Regional",
    "Murrindindi Shire Council":       "Rural & Regional",
    "Northern Grampians Shire Council":"Rural & Regional",
    "Pyrenees Shire Council":          "Rural & Regional",
    "Queenscliffe Borough Council":    "Rural & Regional",
    "South Gippsland Shire Council":   "Rural & Regional",
    "Southern Grampians Shire Council":"Rural & Regional",
    "Strathbogie Shire Council":       "Rural & Regional",
    "Surf Coast Shire Council":        "Rural & Regional",
    "Swan Hill Rural City Council":    "Rural & Regional",
    "Towong Shire Council":            "Rural & Regional",
    "Wellington Shire Council":        "Rural & Regional",
    "West Wimmera Shire Council":      "Rural & Regional",
    "Yarriambiack Shire Council":      "Rural & Regional",
}

council_df['region_type'] = council_df['council'].map(council_region_type_mapping)

In [13]:
gdf = gpd.read_file("Dataset/LGA_POLYGON/LGA_POLYGON.shp")

# convert to WGS84 (lat/long)
gdf = gdf.to_crs(epsg=4326)

# compute centroid
gdf["longitude"] = gdf.geometry.centroid.x
gdf["latitude"] = gdf.geometry.centroid.y

lga_coords = gdf[["LGA_NAME", "latitude", "longitude"]]

lga_coords['LGA_NAME'] = lga_coords['LGA_NAME'].str.title()

council_df = council_df.merge(
    lga_coords,
    how="left",
    left_on='lga',
    right_on='LGA_NAME'
)
council_df = council_df.drop(columns=['LGA_NAME'])

C:\Users\User\AppData\Local\Temp\ipykernel_44168\1749821101.py:7: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["longitude"] = gdf.geometry.centroid.x
C:\Users\User\AppData\Local\Temp\ipykernel_44168\1749821101.py:8: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  gdf["latitude"] = gdf.geometry.centroid.y


In [14]:
# export cleaned dataset
council_df.to_csv('Dataset/Semantic Model/dim_council.csv', index=0)

### Dim Table - Material

In [15]:
material_df = (
    vwp_df[['material_type', 'material_name']]
    .drop_duplicates()
    .sort_values(['material_type', 'material_name'])
    .reset_index(drop=True)
)

material_df['material_id'] = range(1, len(material_df) + 1)

In [16]:
material_df.to_csv('Dataset/Semantic Model/dim_material.csv', index=0)

### Dim Table - Source Sector

In [17]:
source_sector_df = (
    vwp_df[['source_sector']]
    .drop_duplicates()
    .sort_values('source_sector')
    .reset_index(drop=True)
)

source_sector_mapping = {
    'CND': 'Construction and Demolition',
    'CNI': 'Commercial and Industrial',
    'MSW': 'Municipal Solid Waste',
}
source_sector_df['source_sector_full'] = source_sector_df['source_sector'].map(source_sector_mapping)

In [18]:
source_sector_df.to_csv('Dataset/Semantic Model/dim_source_sector.csv', index=0)

### Fact Table - Waste Flow

In [24]:
waste_flow_df = vwp_df.merge(
    material_df,
    on=['material_type', 'material_name'],
    how='left'
)

waste_flow_df = waste_flow_df.drop(columns=['material_type', 'material_name', 'financial_year'])

In [25]:
waste_flow_df = waste_flow_df.rename(columns={
    'disposal': 'Disposal',
    'international_export': 'International Export',
    'interstate_export': 'Interstate Export',
    'processed_locally': 'Processed Locally',
    'total_generation': 'Total Generation'
})


waste_flow_df = waste_flow_df.melt(
    id_vars=['start_year', 'source_sector', 'material_id'],  
    value_vars=[
        'Disposal',
        'International Export',
        'Interstate Export',
        'Processed Locally',
        'Total Generation'
    ],
    var_name='waste_flow',
    value_name='tonnes'
)

In [26]:
waste_flow_df.to_csv('Dataset/Semantic Model/fact_waste_flow.csv', index=0)